In [52]:
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.preprocessing import LabelEncoder
from pathlib import Path
import re

In [53]:
def read_data_set(file_path) -> pd.DataFrame:
    """Reads a CSV file and returns a pandas DataFrame."""
    file_path = Path(file_path)
    return pd.read_csv(file_path)

In [ ]:
# ========== 1. 读取数据 ==========
print("Reading data...")
# data = read_data_set(r'D:\Project\NusSemester1BigDataProjectBETH\G9Proj\data\sample_processes_train.csv')
# validation_data = read_data_set(r'D:\Project\NusSemester1BigDataProjectBETH\G9Proj\data\sample_processes_valid.csv')

data = read_data_set(r'D:\Project\NusSemester1BigDataProjectBETH\G9Proj\data\processed_train_data_all2Number_delProcessname_delMode.csv')
validation_data = read_data_set(r'D:\Project\NusSemester1BigDataProjectBETH\G9Proj\data\processed_valid_data_all2Number_delProcessname_delMode.csv')

# Display initial information
print("Initial training data shape:", data.shape)
print("\nFirst 5 rows of training data:")
print(data.head())
print("\nMissing values in training data:")
print(data.isnull().sum().loc[lambda x: x > 0]) # Only show columns with missing values

# 判断validatation_transfer_data和transfer_data的列是否一致
assert validation_data.columns.tolist() == data.columns.tolist(), "Columns in training and validation data do not match!"

Reading data...
Initial training data shape: (127744, 167)

First 5 rows of training data:
   index  target   timestamp  processId  threadId  parentProcessId  userId  \
0      0       0  124.439221        381       381                1     101   
1      2       0  124.439958          1         1                0       0   
2      4       0  124.440037          1         1                0       0   
3      6       0  124.440379          1         1                0       0   
4      7       0  124.440414          1         1                0       0   

   mountNamespace  hostName  eventId  ...  flag_O_LARGEFILE.3  \
0      4026532232         0       41  ...                   0   
1      4026531840         0     1005  ...                   1   
2      4026531840         0        5  ...                   0   
3      4026531840         0     1005  ...                   1   
4      4026531840         0      257  ...                   0   

   flag_O_NOATIME.3  flag_O_NOCTTY.3  flag_O_NOFO

AssertionError: Columns in training and validation data do not match!

In [55]:
#查看data中是否存在空值
print("\nChecking for null values in training data:")
print(data.isnull().sum())


Checking for null values in training data:
index              0
target             0
timestamp          0
processId          0
threadId           0
                  ..
flag_O_PATH.3      0
flag_O_RDONLY.3    0
flag_O_RDWR.3      0
flag_O_TRUNC.3     0
flag_O_WRONLY.3    0
Length: 167, dtype: int64


In [56]:
data.head()

,index,target,timestamp,processId,threadId,parentProcessId,userId,mountNamespace,hostName,eventId,...,flag_O_LARGEFILE.3,flag_O_NOATIME.3,flag_O_NOCTTY.3,flag_O_NOFOLLOW.3,flag_O_NONBLOCK.3,flag_O_PATH.3,flag_O_RDONLY.3,flag_O_RDWR.3,flag_O_TRUNC.3,flag_O_WRONLY.3
0,0,0,124.439221,381,381,1,101,4026532232,0,41,...,0,0,0,0,0,0,0,0,0,0
1,2,0,124.439958,1,1,0,0,4026531840,0,1005,...,1,0,0,0,0,0,1,0,0,0
2,4,0,124.440037,1,1,0,0,4026531840,0,5,...,0,0,0,0,0,0,0,0,0,0
3,6,0,124.440379,1,1,0,0,4026531840,0,1005,...,1,0,0,0,0,0,1,0,0,0
4,7,0,124.440414,1,1,0,0,4026531840,0,257,...,0,0,0,0,0,0,1,0,0,0


In [57]:
def transfer_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    清理特征集:
    1. 按照BETH中的论文方法处理特征。
    2. 移除因独热编码等产生的重复特征。
    """
    print("  - Cleaning features...")

    # processId: 如果ID是0, 1, or 2，则为1，否则为0
    df['processId_is_os'] = df['processId'].isin([0, 1, 2]).astype(int)
    # df = df.drop(columns='processId')
    # parentProcessId: 规则同processId
    df['parentProcessId_is_os'] = df['parentProcessId'].isin([0, 1, 2]).astype(int)
    # df = df.drop(columns='parentProcessId')
    # userId: 如果ID < 1000，则为1 (系统活动)，否则为0 (用户活动)
    df['userId_is_os'] = (df['userId'] < 1000).astype(int)
    # df = df.drop(columns='userId')
    # mountNamespace: 如果值为 4026531840，则为1，否则为0
    df['mountNamespace_is_default'] = (df['mountNamespace'] == 4026531840).astype(int)
    # df = df.drop(columns='mountNamespace')
    # returnValue: 映射为-1, 0, 1三种状态
    # <0 映射为 -1 (error)
    # =0 映射为 0 (success)
    # >0 映射为 1 (success with signal)
    df['returnValue_mapped'] = df['returnValue'].apply(lambda x: -1 if x < 0 else (0 if x == 0 else 1))
    # df = df.drop(columns='returnValue')

    # 删除hostName特征
    # df = df.drop(columns=['hostName'])
    
    """ # 步骤 1: 移除高基数ID特征
    ids_to_drop = ['processId', 'threadId']
    existing_ids_to_drop = [col for col in ids_to_drop if col in df.columns]
    if existing_ids_to_drop:
        df = df.drop(columns=existing_ids_to_drop)
        print(f"    Removed ID features: {existing_ids_to_drop}") """

    # 步骤 2: 移除重复的Flag特征
    seen_base_names = set()
    cols_to_drop_redundant = []
    for col in df.columns:
        # 使用正则表达式获取基础名称, 例如 'flag_O_RDWR.1' -> 'flag_O_RDWR'
        base_name = re.sub(r'\.\d+$', '', col)
        if base_name in seen_base_names:
            cols_to_drop_redundant.append(col)
        else:
            seen_base_names.add(base_name)

    if cols_to_drop_redundant:
        df = df.drop(columns=cols_to_drop_redundant)
        print(f"    Removed {len(cols_to_drop_redundant)} redundant feature columns.")
        
    return df

In [58]:
""" transfer_data = transfer_features(data)
validation_transfer_data = transfer_features(validation_data)
print("After feature transfer, training data shape:", transfer_data.shape) """

transfer_data = data
validation_transfer_data = validation_data


In [59]:
transfer_data.head().to_dict()


{'index': {0: 0, 1: 2, 2: 4, 3: 6, 4: 7},
 'target': {0: 0, 1: 0, 2: 0, 3: 0, 4: 0},
 'timestamp': {0: 124.439221,
  1: 124.439958,
  2: 124.440037,
  3: 124.440379,
  4: 124.440414},
 'processId': {0: 381, 1: 1, 2: 1, 3: 1, 4: 1},
 'threadId': {0: 381, 1: 1, 2: 1, 3: 1, 4: 1},
 'parentProcessId': {0: 1, 1: 0, 2: 0, 3: 0, 4: 0},
 'userId': {0: 101, 1: 0, 2: 0, 3: 0, 4: 0},
 'mountNamespace': {0: 4026532232,
  1: 4026531840,
  2: 4026531840,
  3: 4026531840,
  4: 4026531840},
 'hostName': {0: 0, 1: 0, 2: 0, 3: 0, 4: 0},
 'eventId': {0: 41, 1: 1005, 2: 5, 3: 1005, 4: 257},
 'argsNum': {0: 3, 1: 4, 2: 2, 3: 4, 4: 4},
 'returnValue': {0: 15, 1: 0, 2: 0, 3: 0, 4: 12},
 'dev': {0: nan, 1: 5.0, 2: nan, 3: 5.0, 4: nan},
 'inode': {0: nan, 1: 38540.0, 2: nan, 3: 38542.0, 4: nan},
 'fd': {0: nan, 1: nan, 2: 12.0, 3: nan, 4: nan},
 'dirfd': {0: nan, 1: nan, 2: nan, 3: nan, 4: -100.0},
 'processName_enc': {0: 21, 1: 17, 2: 17, 3: 17, 4: 17},
 'eventName_enc': {0: 28, 1: 22, 2: 13, 3: 22, 4: 18},
 

In [67]:
validation_transfer_data.head().to_dict()

# 判断validatation_transfer_data和transfer_data的列是否一致
assert transfer_data.columns.tolist() == validation_transfer_data.columns.tolist(), "Columns in training and validation data do not match!"

AssertionError: Columns in training and validation data do not match!

In [60]:
# ========== 3. 挑选特征 ==========
def select_features(transfer_data: pd.DataFrame) -> pd.DataFrame:
    """
    挑选用于训练的特征。
    """
    print("\nSelecting features...")
    train_features = pd.DataFrame()
    train_features['argsNum'] = transfer_data['argsNum']
    train_features['mountNamespace_is_default'] = transfer_data['mountNamespace_is_default']
    train_features['parentProcessId_is_os'] = transfer_data['parentProcessId_is_os']
    train_features['processId_is_os'] = transfer_data['processId_is_os']
    train_features['returnValue_mapped'] = transfer_data['returnValue_mapped']
    train_features['userId_is_os'] = transfer_data['userId_is_os']
    train_features['eventId'] = transfer_data['eventId']
    train_features['target'] = transfer_data['target']
    # transfer_data['']
    # transfer_data['']
    return train_features

In [61]:
# train_features = select_features(transfer_data)
# validation_features = select_features(validation_transfer_data)
# train_features.head()

train_features = transfer_data
validation_features = validation_transfer_data

In [62]:
# ========== 3. 特征-目标分离 ==========

y_train = train_features['target']
y_validation = validation_features['target']
X_train = train_features.drop(columns=['target'])
X_validation = validation_features.drop(columns=['target'])

""" # Align columns - crucial if train/validation sets have different columns after preprocessing
train_cols = X_train.columns
valid_cols = X_validation.columns
shared_cols = list(set(train_cols) & set(valid_cols))
X_train = X_train[shared_cols]
X_validation = X_validation[shared_cols] """

print(f"\nTraining with {X_train.shape[1]} features.")


Training with 166 features.


In [63]:
train_features.head()

,index,target,timestamp,processId,threadId,parentProcessId,userId,mountNamespace,hostName,eventId,...,flag_O_LARGEFILE.3,flag_O_NOATIME.3,flag_O_NOCTTY.3,flag_O_NOFOLLOW.3,flag_O_NONBLOCK.3,flag_O_PATH.3,flag_O_RDONLY.3,flag_O_RDWR.3,flag_O_TRUNC.3,flag_O_WRONLY.3
0,0,0,124.439221,381,381,1,101,4026532232,0,41,...,0,0,0,0,0,0,0,0,0,0
1,2,0,124.439958,1,1,0,0,4026531840,0,1005,...,1,0,0,0,0,0,1,0,0,0
2,4,0,124.440037,1,1,0,0,4026531840,0,5,...,0,0,0,0,0,0,0,0,0,0
3,6,0,124.440379,1,1,0,0,4026531840,0,1005,...,1,0,0,0,0,0,1,0,0,0
4,7,0,124.440414,1,1,0,0,4026531840,0,257,...,0,0,0,0,0,0,1,0,0,0


In [64]:
# ========== 4. 训练 Isolation Forest 模型 (无监督) ==========
print("\nTraining Isolation Forest model...")
# contamination 指的是数据集中异常点的比例。'auto' 是一个常用选项，
# 也可以根据训练数据中的异常比例进行设置。
contamination_rate = 'auto' # or y_train.value_counts(normalize=True)[1]
if 1 in y_train.value_counts(normalize=True):
    contamination_rate = y_train.value_counts(normalize=True)[1]
    print(f"Using contamination rate from training data: {contamination_rate:.4f}")

iso_forest_model = IsolationForest(
    n_estimators=100,
    contamination=contamination_rate,
    # contamination=0.04,
    random_state=42,
    n_jobs=-1 # 使用所有可用的CPU核心
)

# 无监督学习：fit 的时候不需要 y_train
iso_forest_model.fit(X_train)


Training Isolation Forest model...


,n_estimators,100
,max_samples,'auto'
,contamination,'auto'
,max_features,1.0
,bootstrap,False
,n_jobs,-1
,random_state,42
,verbose,0
,warm_start,False


In [65]:
# ========== 5. 模型评估 ==========
print("\nMaking predictions on validation data...")
# predict 返回 -1 (异常) 和 1 (正常)
y_pred = iso_forest_model.predict(X_validation)

# 将预测结果映射到 0 和 1 (0: 正常, 1: 异常) 以便评估
y_pred_mapped = np.where(y_pred == -1, 1, 0)

print("\nModel Evaluation:")
print("Accuracy:", accuracy_score(y_validation, y_pred_mapped))
print("\nConfusion Matrix:")
print(confusion_matrix(y_validation, y_pred_mapped))
print("\nClassification Report:")
print(classification_report(y_validation, y_pred_mapped, zero_division=0))


Making predictions on validation data...


ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- flag_FASYNC
- flag_O_DIRECT
- flag_O_DSYNC
- flag_O_TMPFILE
Feature names seen at fit time, yet now missing:
- flag_-1107596144
- flag_-1107596144.1
- flag_-1107596144.2
- flag_-1107596144.3
- flag_-1369736048
- ...
